<a href="https://colab.research.google.com/github/Melvyn-Bariou/Melvyn-Bariou/blob/main/AL06_Technique_d'IA_2_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP Techniques IA 2 - BUT3
## Agent conversationnel - IUT-RAG

Melvyn BARIOU & Nicolas JOUIN--DERRIEN - BUT3A1

### Installation des dépendances

In [ ]:
!pip install transformers torch accelerate
!pip install lightrag-hku

### Récupération du modèle Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login
mon_token = userdata.get('HF_TOKEN')
login(token=mon_token)

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_id = "google/gemma-2-2b-it"

print("Début du téléchargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Début du téléchargement du modèle...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
print("Succès ! Le modèle est chargé et prêt à être utilisé.")

Début du téléchargement du tokenizer...
Début du téléchargement du modèle...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Succès ! Le modèle est chargé et prêt à être utilisé.


### Intégration dans LightRAG

In [ ]:
import os
import shutil
import nest_asyncio
nest_asyncio.apply()

from lightrag import LightRAG
from lightrag.llm.hf import hf_model_complete, hf_embed
from lightrag.utils import EmbeddingFunc
from transformers import AutoTokenizer, AutoModel

WORKING_DIR = "./iut_rag"

if os.path.exists(WORKING_DIR):
    shutil.rmtree(WORKING_DIR)
os.mkdir(WORKING_DIR)

print("Chargement du modèle de vectorisation BGE-M3 sur CPU (pour économiser la VRAM)...")
embed_model_name = "BAAI/bge-m3"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
embed_model = AutoModel.from_pretrained(embed_model_name).to("cpu")

print("Initialisation de LightRAG...")
rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=hf_model_complete,
    llm_model_name="google/gemma-2-2b-it",
    embedding_func=EmbeddingFunc(
        embedding_dim=1024,
        max_token_size=8192,
        func=lambda texts: hf_embed(texts, tokenizer=embed_tokenizer, embed_model=embed_model)
    )
)

print("Initialisation des bases de stockage internes...")
await rag.initialize_storages()

Chargement du modèle de vectorisation BGE-M3 sur CPU (pour économiser la VRAM)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Initialisation de LightRAG...


INFO: [] Created new empty graph file: ./iut_rag/graph_chunk_entity_relation.graphml
INFO: [] Process 17248 KV load full_docs with 0 records
INFO: [] Process 17248 KV load text_chunks with 0 records
INFO: [] Process 17248 KV load full_entities with 0 records
INFO: [] Process 17248 KV load full_relations with 0 records
INFO: [] Process 17248 KV load entity_chunks with 0 records
INFO: [] Process 17248 KV load relation_chunks with 0 records
INFO: [] Process 17248 KV load llm_response_cache with 0 records
INFO: [] Process 17248 doc status load doc_status with 0 records


Initialisation des bases de stockage internes...


### Définition de la base de connaissances

In [ ]:
# Pour désactiver la celulle utilisé pour la création de la base de connaissances
%%script false --no-raise-error

iut_knowledge = """
L'IUT de Lorient est un Institut Universitaire de Technologie rattaché à l'Université Bretagne Sud.
Il propose des formations en BUT (Bachelor Universitaire de Technologie) sur 3 ans.
Les départements présents à l'IUT de Lorient sont notamment : Informatique, Réseaux et Télécommunications, Gestion des Entreprises et des Administrations, Techniques de Commercialisation.
Le département Informatique propose un BUT Informatique avec plusieurs parcours : développement logiciel, déploiement d'applications, data science.
Le BUT Informatique est accessible après le baccalauréat. La sélection se fait via Parcoursup.
L'IUT propose également des formations en alternance. Les étudiants peuvent réaliser leur BUT en apprentissage.
Les locaux de l'IUT de Lorient se trouvent à Lorient, en Bretagne.
L'IUT de Lorient dispose de laboratoires informatiques équipés pour les travaux pratiques.
Le responsable pédagogique du BUT Informatique accompagne les étudiants tout au long de leur cursus.
Les débouchés du BUT Informatique incluent : développeur logiciel, administrateur systèmes et réseaux, analyste de données, chef de projet informatique.
Les étudiants peuvent poursuivre en master après le BUT.
L'IUT organise chaque année des journées portes ouvertes pour présenter ses formations aux lycéens.
Le BUT3 Informatique comprend des UE (Unités d'Enseignement) en IA, réseaux, développement web, et gestion de projet.
Les stages en entreprise sont obligatoires dans le cursus du BUT Informatique.
"""

print("Insertion des connaissances locales en cours (création du graphe)...")
rag.insert(iut_knowledge)
print("Base de connaissances locales ajoutée avec succès !")

#### Crawler
L'objectif du crawler est d'allé récupérer des données sur une page web pour nourrir le RAG.

In [ ]:
import requests
from bs4 import BeautifulSoup
import nest_asyncio
nest_asyncio.apply()

def scraper_page_iut(url):
    """Télécharge une page web et extrait son texte utile en filtrant le bruit."""
    print(f"Scraping en cours : {url}")
    try:
        reponse = requests.get(url)
        reponse.raise_for_status()

        soup = BeautifulSoup(reponse.text, 'html.parser')
        for element in soup(["nav", "footer", "header", "aside", "script", "style"]):
            element.decompose()

        zone_principale = soup.find('main')
        if not zone_principale:
            zone_principale = soup.body
        textes_extraits = []
        if zone_principale:
            for element in zone_principale.find_all(['h1', 'h2', 'h3', 'p', 'li']):
                texte = element.get_text(strip=True)
                if len(texte) > 40:
                    textes_extraits.append(texte)

        texte_final = "\n".join(textes_extraits)
        return texte_final

    except Exception as e:
        print(f"Erreur lors du scraping : {e}")
        return ""

urls_iut = [
    "https://www.iutvannes.fr/b-u-t-informatique/",
]

texte_global_iut = ""
for url in urls_iut:
    contenu = scraper_page_iut(url)
    texte_global_iut += contenu + "\n\n"

print(f"\nScraping terminé ! {len(texte_global_iut)} caractères récupérés.")

if len(texte_global_iut) > 0:
    print("Injection des nouvelles données dans la base LightRAG...")
    rag.insert(texte_global_iut)
    print("Nouvelles connaissances assimilées avec succès !")

Scraping en cours : https://www.iutvannes.fr/b-u-t-informatique/


INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-7b64eeb028a38e332375a0b5c55286aa
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)



Scraping terminé ! 10086 caractères récupérés.
Injection des nouvelles données dans la base LightRAG...


INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
INFO:  == LLM cache == saving: default:extract:a341ad6f2a940cd38ddbf7a1d2cf701a
INFO:  == LLM cache == saving: default:extract:3fd756118d0f0c83ee952ff9a9860833
INFO:  == LLM cache == saving: default:extract:c0d5c906c42020301486569e377a0976
INFO:  == LLM cache == saving: default:extract:d9b3e380a073daa1b1b5f3e92bd802ae
INFO:  == LLM cache == saving: default:extract:1876baa09cf4c28be359f1122e9d9def
INFO:  == LLM cache == saving: default:extract:37c457934b8a97afda9ef8d1b902a34c
INFO: Chunk 1 of 3 extracted 9 Ent + 0 Rel chunk-1e7f153b2dc0764ba8d43f4662c5c621
INFO: Chunk 2 of 3 extracted 15 Ent + 0 Rel chunk-71b5b168b0e8574abe47245ba9e0b5ad
INFO: Chunk 3 of 3 extracted 13 Ent + 0 Rel chunk-5162615ad328aa7b2cd9e9464b32a9d3
INFO: Merging stage 1/1: unknown_source
INFO: Phase 1: Processing 35 entities from doc-7b64eeb028a38e332375a0b5c55286aa (async: 8)
INFO

Nouvelles connaissances assimilées avec succès !


### Nettoyage de la mémoire du GPU
Nous sommes obligé de réaliser cette opération de nettoyage étant donné des limitations de collab pour pouvoir libérer la mémoire afin de permettre de poser des questions.

In [ ]:
import torch
import gc

if 'model' in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

### Poser une question au modèle

In [ ]:
question = "Est-ce que tu connais des écoles d'ingé ?"

try:
    # Exécution de la requête avec gestion d'erreur
    response = rag.query(question)
    print("Question :", question)
    print("Réponse :", response)
except Exception as e:
    print(f"Erreur lors de la requête : {e}")

INFO:  == LLM cache == saving: mix:keywords:47c9e77db9902f691a5336e41944b2e7
INFO: Query nodes: écoles, ingénieurs (top_k:40, cosine:0.2)
INFO: Local query: 35 entites, 0 relations
INFO: Query edges: écoles d'ingénieurs (top_k:40, cosine:0.2)
INFO: Naive query: 3 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 35 entities, 0 relations, 3 vector chunks
INFO: After truncation: 35 entities, 0 relations
INFO: Selecting 3 from 3 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 6 -> 3 (deduplicated 3)
INFO: Final context: 35 entities, 0 relations, 3 chunks
INFO: Final chunks S+F/O: E13/1 E9/2 E15/3
INFO:  == LLM cache == saving: mix:query:982e7c32b3ab6c73508fd7d198177609


Question : Est-ce que tu connais des écoles d'ingé ?
Réponse : The IUT de Vannes offers a Bachelor's degree in Computer Science, which is a good starting point for a career in computer science. 

Here are some other schools of engineering in France:

* **École Polytechnique (X) :**  A highly selective engineering school known for its rigorous curriculum and focus on mathematics and physics.
* **École Nationale Supérieure des Mines de Saint-Étienne (Mines Saint-Étienne) :** A leading engineering school specializing in mining, energy, and materials science.
* **École Centrale Paris :** A prestigious engineering school with a strong focus on technology and innovation.
* **École des Mines de Saint-Étienne (Mines Saint-Étienne) :** A leading engineering school specializing in mining, energy, and materials science.
* **École Nationale Supérieure des Arts et Métiers (ENSAM) :** A renowned engineering school with a focus on applied sciences and technology.
* **École des Ponts ParisTech :** A h